In [ ]:
!pip install pymupdf -q
!pip install sentence-transformers -q
!pip install chromadb -q
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.8 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Motion.pdf to Motion.pdf
Saving priodic_table.pdf to priodic_table.pdf


In [ ]:
import os
for f in os.listdir():
    print(f)

.config
Motion.pdf
priodic_table.pdf
sample_data


In [ ]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ""
    return text

In [ ]:
motion_text = extract_text_from_pdf("Motion.pdf")
periodic_text = extract_text_from_pdf("priodic_table.pdf")

print("Motion text length:", len(motion_text))
print(motion_text[:500])
print("\n---\n")
print("Periodic table text length:", len(periodic_text))
print(periodic_text[:500])

Motion text length: 67654
Chapter 3
Motion in One Dimension - Grade
10
3.1 Introduction
This chapter is about how things move in a straight line or more scientiﬁcally how things move in
one dimension . This is useful for learning how to describe the movement of cars along a straight
road or of trains along straight railway tracks. If you want to understand how any object moves,
for example a car on the freeway, a soccer ball being kicked towards the goal or your dog chasing
the neighbour’s cat, then you have to understan

---

Periodic table text length: 29392
Periodic Classification
of Elements5 CHAPTER
In Class IX we have learnt that matter around us is present in the form
of elements, compounds and mixtures and the elements contain atoms
of only one type. Do you know how many elements are known till date?
At present, 118 elements are known to us. All these have different
properties. Out of these 118, only 94 are naturally occurring.
As different elements were being discovered, scien

In [ ]:
print("Motion text length:", len(motion_text))
print("Periodic table text length:", len(periodic_text))
# If periodic table text is short, it's likely image-based so, we use OCR
if len(periodic_text.strip()) < 100:
    print("\nPeriodic table looks image-based. Installing OCR tools...")
    !pip install pytesseract pdf2image -q
    !apt-get install poppler-utils tesseract-ocr -q

    import pytesseract
    from pdf2image import convert_from_path

    pages = convert_from_path("priodic_table.pdf")
    periodic_text = ""
    for page_image in pages:
        periodic_text += pytesseract.image_to_string(page_image)

    print("OCR done. New periodic table text length:", len(periodic_text))
else:
    print("\nPeriodic table extracted normally, No OCR needed.")

print("\nPreview of periodic table text:")
print(periodic_text[:500])

Motion text length: 67654
Periodic table text length: 29392

Periodic table extracted normally, no OCR needed.

Preview of periodic table text:
Periodic Classification
of Elements5 CHAPTER
In Class IX we have learnt that matter around us is present in the form
of elements, compounds and mixtures and the elements contain atoms
of only one type. Do you know how many elements are known till date?
At present, 118 elements are known to us. All these have different
properties. Out of these 118, only 94 are naturally occurring.
As different elements were being discovered, scientists gathered more
and more information about the properties of th


In [ ]:
!pip install langchain-text-splitters -q

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

motion_chunks = splitter.split_text(motion_text)
periodic_chunks = splitter.split_text(periodic_text)

print(f"Motion PDF: {len(motion_chunks)} chunks")
print(f"Periodic Table PDF: {len(periodic_chunks)} chunks")

# Info about sample chunk
print("\nSample chunk from periodic table:")
print(periodic_chunks[0])

Motion PDF: 149 chunks
Periodic Table PDF: 64 chunks

Sample chunk from periodic table:
Periodic Classification
of Elements5 CHAPTER
In Class IX we have learnt that matter around us is present in the form
of elements, compounds and mixtures and the elements contain atoms
of only one type. Do you know how many elements are known till date?
At present, 118 elements are known to us. All these have different
properties. Out of these 118, only 94 are naturally occurring.
As different elements were being discovered, scientists gathered more


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(name="pdf_docs")

def add_chunks_to_db(chunks, source_name):
    embeddings = model.encode(chunks).tolist()
    ids = [f"{source_name}_{i}" for i in range(len(chunks))]
    metadatas = [{"source": source_name} for _ in chunks]

    collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=ids,
        metadatas=metadatas
    )
    print(f"Added {len(chunks)} chunks from '{source_name}'")

add_chunks_to_db(motion_chunks, "motion")
add_chunks_to_db(periodic_chunks, "periodic_table")

print("\nTotal items in database:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Added 149 chunks from 'motion'
Added 64 chunks from 'periodic_table'

Total items in database: 213


In [ ]:
sample = collection.peek(limit=3)
for doc, meta in zip(sample['documents'], sample['metadatas']):
    print(f"[{meta['source']}] {doc[:150]}...")
    print()

[motion] Chapter 3
Motion in One Dimension - Grade
10
3.1 Introduction
This chapter is about how things move in a straight line or more scientiﬁcally how thing...

[motion] the neighbour’s cat, then you have to understand three basic ideas about what it means when
something is moving . These three ideas describe diﬀerent ...

[motion] You will also learn how to use position, displacement, speed, velocity and acceleration to describe
the motion of simple objects. You will learn how t...



In [ ]:
def search(query, n_results=3, source_filter=None):
    query_embedding = model.encode([query]).tolist()

    kwargs = {"query_embeddings": query_embedding, "n_results": n_results}
    if source_filter:
        kwargs["where"] = {"source": source_filter}

    results = collection.query(**kwargs)

    for i in range(len(results['documents'][0])):
        distance = results['distances'][0][i]
        similarity = 1 - distance
        source = results['metadatas'][0][i]['source']
        print(f"--- Result {i+1} | source: {source} | similarity: {similarity:.3f} ---")
        print(results['documents'][0][i])
        print()

# Test with real questions
search("what is Newton's first law of motion")
print("=" * 60)
search("atomic number of oxygen")

--- Result 1 | source: motion | similarity: 0.225 ---
unless a force – often friction – acts upon them, refuting the accepted Aristotelian
hypothesis that objects ”naturally” slow down and stop unless a force acts uponthem. This principle was incorporated into Newton’s laws of motion (1st law).
3.9.1 Finding the Equations of Motion
The following does not form part of the syllabus and can be considered additional information.
54CHAPTER 3. MOTION IN ONE DIMENSION - GRADE 10 3.9
Derivation of Equation 3.1
According to the deﬁnition of acceleration:

--- Result 2 | source: motion | similarity: -0.134 ---
a constant acceleration (motion at constant acceleration).
393.6 CHAPTER 3. MOTION IN ONE DIMENSION - GRADE 10
3.6.1 Stationary Object
The simplest motion that we can come across is that of a stationary object. A stationary object
does not move and so its position does not change, for as long as it is standing still. An exampleof this situation is when someone is waiting for something with

In [ ]:
# Delete the old collection (it was created with the wrong distance metric)
client.delete_collection(name="pdf_docs")

# Recreating it, explicitly telling ChromaDB to use COSINE distance
collection = client.get_or_create_collection(
    name="pdf_docs",
    metadata={"hnsw:space": "cosine"}  # this tells ChromaDB to use cosine similarity math
)

# Re-add everything (since we deleted the collection, it's now empty)
add_chunks_to_db(motion_chunks, "motion")
add_chunks_to_db(periodic_chunks, "periodic_table")

print("Total items in database:", collection.count())

Added 149 chunks from 'motion'
Added 64 chunks from 'periodic_table'
Total items in database: 213


In [ ]:
def search(query, n_results=3, source_filter=None):
    query_embedding = model.encode([query]).tolist()

    kwargs = {"query_embeddings": query_embedding, "n_results": n_results}
    if source_filter:
        kwargs["where"] = {"source": source_filter}

    results = collection.query(**kwargs)

    for i in range(len(results['documents'][0])):
        distance = results['distances'][0][i]
        similarity = 1 - distance
        source = results['metadatas'][0][i]['source']
        print(f"--- Result {i+1} | source: {source} | similarity: {similarity:.3f} ---")
        print(results['documents'][0][i])
        print()

# Re-test
search("what is Motion")
print("=" * 60)
search("What is frame of reference")

--- Result 1 | source: motion | similarity: 0.576 ---
The most important idea when studying motion, is you have to know where you are. The
word position describes your location (where you are). However, saying that you are hereis
meaningless, and you have to specify your position relative to a known reference point. For
example, if you are 2 m from the doorway, inside your classroom then your reference point is
the doorway. This deﬁnes your position inside the classroom. Notice that you need a reference

--- Result 2 | source: motion | similarity: 0.563 ---
3.12 End of Chapter Exercises: Motion in One Dimension
1. Give one word/term for the following descriptions.
(a) The shortest path from start to ﬁnish.
(b) A physical quantity with magnitude and direction.
(c) The quantity deﬁned as a change in velocity over a time period.
(d) The point from where you take measurements.
(e) The distance covered in a time interval.
(f) The velocity at a speciﬁc instant in time.

--- Result 3 | source

In [1]:
# Implementing LLM
!pip install groq -q

In [ ]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter Your api key:")

Enter Your api key:··········


In [ ]:
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def ask_llm(question, context):
    """
    Sends the retrieved context + question to Groq's LLM,
    asking it to answer ONLY using the provided context.
    """
    prompt = f"""Answer the question using ONLY the context below.
If the answer isn't in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,  # low temperature = more factual, less "creative"
    )

    return response.choices[0].message.content

In [ ]:
def rag_answer(question, n_results=3, source_filter=None):
    """
    The full RAG pipeline:
    1. Search ChromaDB for relevant chunks
    2. Combine those chunks into context
    3. Send context + question to the LLM
    4. Return a clean, grounded answer
    """
    query_embedding = model.encode([question]).tolist()

    kwargs = {"query_embeddings": query_embedding, "n_results": n_results}
    if source_filter:
        kwargs["where"] = {"source": source_filter}

    results = collection.query(**kwargs)

    # Combine the top retrieved chunks into one context block
    context = "\n\n".join(results['documents'][0])

    # Send to LLM
    answer = ask_llm(question, context)

    print("Question:", question)
    print("\nRetrieved context used:")
    print(context[:300], "...\n")
    print("LLM Answer:")
    print(answer)

    return answer

# Test it!
rag_answer("what is Newton's first law of motion")

Question: what is Newton's first law of motion

Retrieved context used:
unless a force – often friction – acts upon them, refuting the accepted Aristotelian
hypothesis that objects ”naturally” slow down and stop unless a force acts uponthem. This principle was incorporated into Newton’s laws of motion (1st law).
3.9.1 Finding the Equations of Motion
The following does n ...

LLM Answer:
Newton's first law of motion states that unless a force (often friction) acts upon an object, it will not slow down or stop, refuting the Aristotelian hypothesis that objects "naturally" slow down and stop unless a force acts upon them.


'Newton\'s first law of motion states that unless a force (often friction) acts upon an object, it will not slow down or stop, refuting the Aristotelian hypothesis that objects "naturally" slow down and stop unless a force acts upon them.'